In [1]:
import os
import numpy as np
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold


# ======================================================
# 1) BMS feature extraction from each filename (00001.csv, etc.)
# ======================================================

def compute_bms_features_for_file(file_path: str):
    """
    Compute engineered features from one cycle's BMS time series.

    Expects columns in the CSV:
        Voltage_measured, Current_measured, Temperature_measured,
        Current_load, Voltage_load, Time

    Returns a dict with BMS-based features.
    If file does not exist or fails, returns NaNs for all features.
    """
    feature_names = [
        "bms_disc_capacity_ah",
        "bms_charge_capacity_ah",
        "bms_active_time_s",
        "bms_mean_temp",
        "bms_max_temp",
        "bms_mean_voltage",
        "bms_min_voltage",
        "bms_max_voltage",
        "bms_mean_current",
        "bms_max_current",
    ]

    if not os.path.isfile(file_path):
        return {name: np.nan for name in feature_names}

    try:
        df = pd.read_csv(file_path)
    except Exception:
        return {name: np.nan for name in feature_names}

    required_cols = [
        "Voltage_measured", "Current_measured", "Temperature_measured",
        "Current_load", "Voltage_load", "Time"
    ]
    if not all(col in df.columns for col in required_cols):
        return {name: np.nan for name in feature_names}

    df = df.sort_values("Time").reset_index(drop=True)
    df["delta_t"] = df["Time"].diff().fillna(0.0)

    # active = when load is actually applied
    load_threshold = 0.5
    active = df["Current_load"] > load_threshold
    df["delta_t_active"] = np.where(active, df["delta_t"], 0.0)

    # Current cleaning
    I = df["Current_measured"].astype(float).values
    current_noise_threshold = 1e-3
    I_clean = np.where(np.abs(I) < current_noise_threshold, 0.0, I)

    # Separate discharge (<0) and charge (>0) currents
    I_discharge = np.clip(I_clean, None, 0.0)  # <= 0
    I_charge = np.clip(I_clean, 0.0, None)     # >= 0

    # Ah during active periods
    disc_Ah = (-I_discharge * df["delta_t_active"].values / 3600.0).sum()
    chg_Ah = (I_charge * df["delta_t_active"].values / 3600.0).sum()

    # Restrict stats to active time where dt_active > 0
    active_mask = (df["delta_t_active"] > 0)
    active_rows = df[active_mask] if active_mask.any() else df

    mean_temp = float(active_rows["Temperature_measured"].mean())
    max_temp = float(active_rows["Temperature_measured"].max())
    mean_voltage = float(active_rows["Voltage_measured"].mean())
    min_voltage = float(active_rows["Voltage_measured"].min())
    max_voltage = float(active_rows["Voltage_measured"].max())
    mean_current = float(active_rows["Current_measured"].mean())
    max_current = float(active_rows["Current_measured"].max())
    active_time_s = float(df["delta_t_active"].sum())

    return {
        "bms_disc_capacity_ah": disc_Ah,
        "bms_charge_capacity_ah": chg_Ah,
        "bms_active_time_s": active_time_s,
        "bms_mean_temp": mean_temp,
        "bms_max_temp": max_temp,
        "bms_mean_voltage": mean_voltage,
        "bms_min_voltage": min_voltage,
        "bms_max_voltage": max_voltage,
        "bms_mean_current": mean_current,
        "bms_max_current": max_current,
    }


def add_bms_features_to_metadata(metadata: pd.DataFrame, data_dir: str) -> pd.DataFrame:
    """
    For each row in metadata, read its 'filename' CSV from data_dir
    and add BMS-based features as new columns.
    """
    bms_feature_names = [
        "bms_disc_capacity_ah",
        "bms_charge_capacity_ah",
        "bms_active_time_s",
        "bms_mean_temp",
        "bms_max_temp",
        "bms_mean_voltage",
        "bms_min_voltage",
        "bms_max_voltage",
        "bms_mean_current",
        "bms_max_current",
    ]

    for name in bms_feature_names:
        if name not in metadata.columns:
            metadata[name] = np.nan

    for idx, row in metadata.iterrows():
        fname = row.get("filename", None)
        if not isinstance(fname, str):
            continue

        fpath = os.path.join(data_dir, fname)
        feats = compute_bms_features_for_file(fpath)

        for k, v in feats.items():
            metadata.at[idx, k] = v

    return metadata


# ======================================================
# 2) Training + filling a single target (Capacity / Re / Rct)
# ======================================================

def train_and_fill_target(
    metadata: pd.DataFrame,
    target_col: str,
    feature_cols: list,
    categorical_cols: list,
    random_state: int = 42,
):
    """
    Train a regression model for `target_col` using rows where it's present,
    then predict missing values.

    - Uses HistGradientBoostingRegressor (tree-based, strong baseline)
    - Handles numeric + categorical features
    - Imputes missing features internally
    - Prints basic cross-val scores (MAE, R^2)

    Returns:
        metadata with a new column:
            f"{target_col}_pred"
        and the trained model.
    """
    df = metadata.copy()

    # Ensure target is numeric
    df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

    # Feature matrix
    X = df[feature_cols].copy()
    y = df[target_col]

    # Train only on rows where target is known
    mask_train = y.notna()
    X_train = X[mask_train]
    y_train = y[mask_train]

    if X_train.empty:
        print(f"[WARN] No training data available for target '{target_col}'.")
        df[f"{target_col}_pred"] = df[target_col]
        return df, None

    # Numeric features = rest of feature_cols
    numeric_cols = [c for c in feature_cols if c not in categorical_cols]

    # --- Preprocessing: force DENSE output so HGBR is happy ---
    numeric_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    # Handle sklearn version difference for OneHotEncoder dense output
    ohe_kwargs = {"handle_unknown": "ignore"}
    ver = tuple(int(x) for x in sklearn.__version__.split(".")[:2])
    if ver >= (1, 2):
        ohe_kwargs["sparse_output"] = False  # sklearn >= 1.2
    else:
        ohe_kwargs["sparse"] = False         # sklearn < 1.2

    categorical_transformer = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(**ohe_kwargs)),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ],
        sparse_threshold=0.0,  # force dense
    )

    # Regressor – gradient boosting (needs dense)
    regressor = HistGradientBoostingRegressor(
        random_state=random_state,
        max_depth=None,
        max_iter=300,
        learning_rate=0.05,
    )

    model = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("regressor", regressor),
        ]
    )

    # --- Cross-validation (guard for small sample sizes) ---
    n_train = len(X_train)
    n_splits = min(5, n_train)

    if n_splits > 1:
        cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

        mae_scores = -cross_val_score(
            model, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error"
        )
        r2_scores = cross_val_score(
            model, X_train, y_train, cv=cv, scoring="r2"
        )

        print(f"Target: {target_col}")
        print(f"  MAE ({n_splits}-fold CV): {mae_scores.mean():.4f} ± {mae_scores.std():.4f}")
        print(f"  R²  ({n_splits}-fold CV): {r2_scores.mean():.4f} ± {r2_scores.std():.4f}")
    else:
        print(f"Target: {target_col} – only {n_train} labeled samples, skipping CV.")

    # Fit on all labeled data
    model.fit(X_train, y_train)

    # Predict for missing targets
    mask_missing = y.isna()
    df[f"{target_col}_pred"] = df[target_col]

    if mask_missing.any():
        X_missing = X[mask_missing]
        y_pred = model.predict(X_missing)
        df.loc[mask_missing, f"{target_col}_pred"] = y_pred

    return df, model


# ======================================================
# 3) End-to-end usage
# ======================================================

if __name__ == "__main__":
    # ---- Adjust these paths as needed ----
    METADATA_PATH = "/kaggle/input/nasa-battery-dataset/cleaned_dataset/metadata.csv"  # path to your metadata
    DATA_DIR = "/kaggle/input/nasa-battery-dataset/cleaned_dataset/data"                    # folder containing 00001.csv, 00002.csv, ...
    SAVE_PATH = "/kaggle/working/metadata_with_predictions.csv"

    # 1) Load metadata (columns: type, start_time, ambient_temperature,
    #                   battery_id, test_id, uid, filename, Capacity, Re, Rct)
    meta = pd.read_csv(METADATA_PATH)

    # 2) Add BMS-based features from each filename CSV
    meta = add_bms_features_to_metadata(meta, DATA_DIR)

    # 3) Define feature columns for learning
    #    (avoid target columns themselves to prevent leakage)
    feature_cols = [
        "type",
        "ambient_temperature",
        "battery_id",
        "test_id",
        "uid",
        # BMS engineered features:
        "bms_disc_capacity_ah",
        "bms_charge_capacity_ah",
        "bms_active_time_s",
        "bms_mean_temp",
        "bms_max_temp",
        "bms_mean_voltage",
        "bms_min_voltage",
        "bms_max_voltage",
        "bms_mean_current",
        "bms_max_current",
    ]

    # Categorical features among the above
    categorical_cols = ["type", "battery_id"]

    # 4) Train + fill Capacity
    meta, model_capacity = train_and_fill_target(
        metadata=meta,
        target_col="Capacity",
        feature_cols=feature_cols,
        categorical_cols=categorical_cols,
        random_state=42,
    )

    # 5) Train + fill Re
    meta, model_re = train_and_fill_target(
        metadata=meta,
        target_col="Re",
        feature_cols=feature_cols,
        categorical_cols=categorical_cols,
        random_state=42,
    )

    # 6) Train + fill Rct
    meta, model_rct = train_and_fill_target(
        metadata=meta,
        target_col="Rct",
        feature_cols=feature_cols,
        categorical_cols=categorical_cols,
        random_state=42,
    )

    # 7) Save final metadata with predicted values
    meta.to_csv(SAVE_PATH, index=False)
    print(f"\nSaved metadata with predicted Capacity/Re/Rct to: {SAVE_PATH}")

    # Quick sanity check
    cols_to_show = [
        "battery_id", "type", "filename",
        "Capacity", "Capacity_pred",
        "Re", "Re_pred",
        "Rct", "Rct_pred",
    ]
    print(meta[cols_to_show].head(20))


Target: Capacity
  MAE (5-fold CV): 0.0146 ± 0.0016
  R²  (5-fold CV): 0.9920 ± 0.0034
Target: Re
  MAE (5-fold CV): 1416005832248.8608 ± 624712297807.5349
  R²  (5-fold CV): -69180456055299268608.0000 ± 99070618806384263168.0000
Target: Rct
  MAE (5-fold CV): 3004451167705.5518 ± 1325501312981.0132
  R²  (5-fold CV): -1858449081849222004736.0000 ± 3261231583353939427328.0000

Saved metadata with predicted Capacity/Re/Rct to: /kaggle/working/metadata_with_predictions.csv
   battery_id       type   filename  Capacity  Capacity_pred        Re  \
0       B0047  discharge  00001.csv  1.674305       1.674305       NaN   
1       B0047  impedance  00002.csv       NaN       1.105900  0.056058   
2       B0047     charge  00003.csv       NaN       1.402077       NaN   
3       B0047  impedance  00004.csv       NaN       1.107108  0.053192   
4       B0047  discharge  00005.csv  1.524366       1.524366       NaN   
5       B0047     charge  00006.csv       NaN       1.401114       NaN   
6     

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


In [2]:
import os
import shutil
import pandas as pd
import numpy as np

# ======================================================
# 1) SoC function using Capacity_pred + phase split
# ======================================================

def compute_soc_precise(
    df: pd.DataFrame,
    capacity_Ah: float,
    time_col: str = "Time",
    current_col: str = "Current_measured",
    voltage_col: str = "Voltage_measured",
    current_noise_threshold: float = 1e-3,
):
    """
    Estimate State of Charge (SoC) from BMS data using coulomb counting
    and a known capacity (e.g. metadata['Capacity_pred']).

    Assumptions:
      - Time column is in seconds.
      - Current_measured (A): 
          * positive during charge
          * negative during discharge
      - Voltage_measured only used for sanity checks (not required in math).

    Phases:
      - 'charge'     : |I| >= threshold and I > 0
      - 'discharge'  : |I| >= threshold and I < 0
      - 'impedance'  : |I| <  threshold (near zero current)

    Returns:
        df_out: DataFrame with extra columns:
            - delta_t        : time step in seconds
            - I_clean        : current with noise removed
            - phase          : 'charge', 'discharge', 'impedance'
            - delta_Ah       : incremental ΔAh ( >0 charge, <0 discharge )
            - cum_Ah         : cumulative Ah from the first sample
            - SoC_unclipped  : raw SoC (relative, can be outside [0,1])
            - SoC            : SoC clipped to [0,1]
            - SoC_percent    : SoC in %
    """

    df = df.copy()
    df = df.sort_values(time_col).reset_index(drop=True)

    # 1) Time step Δt (seconds)
    df["delta_t"] = df[time_col].diff().fillna(0.0)

    # 2) Noise filter on current
    I = df[current_col].values.astype(float)
    I_clean = np.where(np.abs(I) < current_noise_threshold, 0.0, I)
    df["I_clean"] = I_clean

    # 3) Phase classification
    phase = np.full(len(df), "impedance", dtype=object)
    phase[(I_clean >  current_noise_threshold)] = "charge"
    phase[(I_clean < -current_noise_threshold)] = "discharge"
    df["phase"] = phase

    # 4) Convert current*dt to Ah
    #    Positive I_clean -> positive delta_Ah  (charging)
    #    Negative I_clean -> negative delta_Ah  (discharging)
    df["delta_Ah"] = df["I_clean"] * df["delta_t"] / 3600.0

    # 5) Cumulative Ah from the start of the file
    df["cum_Ah"] = df["delta_Ah"].cumsum()

    # 6) SoC from coulomb counting using known capacity
    capacity_Ah = float(capacity_Ah)

    # Start SoC at 0 for the first row (relative SoC).
    # SoC_unclipped increases during charge and decreases during discharge.
    df["SoC_unclipped"] = df["cum_Ah"] / capacity_Ah

    # 7) Clip SoC to physical range [0,1] for convenience
    df["SoC"] = df["SoC_unclipped"].clip(0.0, 1.0)
    df["SoC_percent"] = df["SoC"] * 100.0

    return df, capacity_Ah


# ======================================================
# 2) Run SoC computation on all CSV files using Capacity_pred
# ======================================================

# Paths – adjust to your setup
METADATA_PATH = "/kaggle/working/metadata_with_predictions.csv"
DATA_DIR      = "/kaggle/input/nasa-battery-dataset/cleaned_dataset/data"
OUTPUT_DIR    = "/kaggle/working/data_with_soc"   # new folder for CSVs with SoC

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load metadata so we know which filenames to process
metadata = pd.read_csv(METADATA_PATH)
metadata["filename"] = metadata["filename"].astype(str)

# Lookups: filename -> Capacity_pred and filename -> type
capacity_map = (
    metadata
    .set_index("filename")["Capacity_pred"]
    .to_dict()
)

type_map = (
    metadata
    .set_index("filename")["type"]
    .to_dict()
)

# Process each file listed in metadata
for i, fname in enumerate(metadata["filename"], start=1):
    in_path = os.path.join(DATA_DIR, fname)
    out_path = os.path.join(OUTPUT_DIR, fname)

    if not os.path.isfile(in_path):
        print(f"[WARN] {fname}: file not found, skipping.")
        continue

    # If this file is of type "impedance", just copy it without modification
    ftype = type_map.get(fname, "")
    if isinstance(ftype, str) and ftype.lower() == "impedance":
        shutil.copyfile(in_path, out_path)
        if i % 100 == 0:
            print(f"Processed {i} files (impedance copied)...")
        continue

    # Get capacity for this file from Capacity_pred
    capacity_Ah = capacity_map.get(fname, None)
    if capacity_Ah is None or np.isnan(capacity_Ah):
        print(f"[WARN] {fname}: Capacity_pred missing/NaN, skipping.")
        continue

    try:
        df = pd.read_csv(in_path)
    except Exception as e:
        print(f"[WARN] {fname}: error reading file ({e}), skipping.")
        continue

    # Make sure required columns exist
    required_cols = ["Time", "Current_measured", "Voltage_measured"]
    if not all(col in df.columns for col in required_cols):
        print(f"[WARN] {fname}: missing required columns {required_cols}, skipping.")
        continue

    # Compute SoC for this file using known capacity from metadata
    df_soc, cap_Ah_used = compute_soc_precise(
        df,
        capacity_Ah=capacity_Ah,
        time_col="Time",
        current_col="Current_measured",
        voltage_col="Voltage_measured",
        current_noise_threshold=1e-3,
    )

    df_soc.to_csv(out_path, index=False)

    # Progress log every 100 files
    if i % 100 == 0:
        print(f"Processed {i} files...")

print("Done. All CSVs with SoC (and raw impedance files) are in:", OUTPUT_DIR)


/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)


Processed 100 files (impedance copied)...


/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)


Processed 200 files (impedance copied)...


/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)


...

Processed 6500 files...
Processed 6600 files...
Processed 6700 files...
Processed 6800 files...


/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)


Processed 6900 files (impedance copied)...
Processed 7000 files...


/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)


Processed 7100 files...
Processed 7200 files...
Processed 7300 files...
Processed 7400 files...
Processed 7500 files...
Done. All CSVs with SoC (and raw impedance files) are in: /kaggle/working/data_with_soc


In [13]:
import pandas as pd
import numpy as np

# ========= CONFIG =========
INPUT_METADATA_PATH  = "/kaggle/working/metadata_with_predictions.csv"
OUTPUT_METADATA_PATH = "/kaggle/working/metadata_with_cycle_no.csv"
# ==========================

# Load metadata
metadata = pd.read_csv(INPUT_METADATA_PATH)

# Basic checks
for col in ["battery_id", "filename", "type"]:
    if col not in metadata.columns:
        raise ValueError(f"Required column '{col}' not found in metadata.")

# Keep track of original row order
metadata["orig_index"] = np.arange(len(metadata))

# Ensure the key columns are the right type
metadata["filename"] = metadata["filename"].astype(str)

# If start_time is a string, convert to datetime for proper ordering
if "start_time" in metadata.columns:
    metadata["start_time"] = pd.to_datetime(metadata["start_time"], errors="coerce")
else:
    # If start_time is missing, we just sort by filename inside each battery
    metadata["start_time"] = pd.NaT

# Create a sorted copy for cycle computation (does NOT affect original order)
meta_sorted = metadata.sort_values(
    ["battery_id", "start_time", "filename"],
    ignore_index=True
)


def assign_cycle_per_battery(group: pd.DataFrame) -> pd.DataFrame:
    """
    For one battery_id, assign cycle_no such that:

    - Each *discharge* step marks the end of a cycle.
    - All rows from the previous discharge (or start) up to this discharge
      belong to the same cycle.
    - Rows after the last discharge are treated as the *next* (possibly incomplete)
      cycle.
    """
    group = group.copy()   # preserve incoming index
    n = len(group)

    cycle = np.zeros(n, dtype=int)

    type_lower = group["type"].astype(str).str.lower()
    is_discharge = type_lower.eq("discharge").to_numpy()
    dis_positions = np.where(is_discharge)[0]

    if len(dis_positions) == 0:
        # No discharge at all: treat everything as cycle 1
        cycle[:] = 1
    else:
        prev_end = -1
        for j, pos in enumerate(dis_positions):
            cyc = j + 1  # cycles start at 1
            start = prev_end + 1
            end = pos
            cycle[start:end + 1] = cyc
            prev_end = pos

        # Anything after the last discharge -> next cycle index
        if prev_end < n - 1:
            cycle[prev_end + 1:] = len(dis_positions) + 1

    # Assign using positional index (group index is preserved)
    group["cycle_no"] = cycle
    return group


# Apply per battery_id on the SORTED copy
meta_sorted_with_cycle = (
    meta_sorted
    .groupby("battery_id", group_keys=False)
    .apply(assign_cycle_per_battery)
)

# We only need orig_index + cycle_no to bring back to original metadata
cycle_map = meta_sorted_with_cycle[["orig_index", "cycle_no"]]

# Merge cycle_no back into original metadata using orig_index
metadata = metadata.merge(cycle_map, on="orig_index", how="left")

# Restore original row order
metadata = metadata.sort_values("orig_index").reset_index(drop=True)

# Drop helper column
metadata = metadata.drop(columns=["orig_index"])

# Save updated metadata
metadata.to_csv(OUTPUT_METADATA_PATH, index=False)

print("Saved metadata with charge–discharge-based cycle_no to:", OUTPUT_METADATA_PATH)


/tmp/ipykernel_48/287624289.py:25: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  metadata["start_time"] = pd.to_datetime(metadata["start_time"], errors="coerce")


Saved metadata with charge–discharge-based cycle_no to: /kaggle/working/metadata_with_cycle_no.csv


/tmp/ipykernel_48/287624289.py:81: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(assign_cycle_per_battery)


In [14]:
df453 = pd.read_csv("/kaggle/working/metadata_with_cycle_no.csv")
df453.head(100)

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct,...,bms_max_temp,bms_mean_voltage,bms_min_voltage,bms_max_voltage,bms_mean_current,bms_max_current,Capacity_pred,Re_pred,Rct_pred,cycle_no
0,discharge,NaN,4,B0047,0,1,00001.csv,1.674305,NaN,NaN,...,12.348018,3.482222,2.470612,4.039277,-0.995350,-0.992155,1.674305,-2.828261e+12,6.000944e+12,1
1,impedance,NaN,24,B0047,1,2,00002.csv,NaN,0.056058,0.200970,...,NaN,NaN,NaN,NaN,NaN,NaN,1.105900,5.605783e-02,2.009702e-01,2
2,charge,NaN,4,B0047,2,3,00003.csv,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.402077,1.125286e+12,-2.387609e+12,2
3,impedance,NaN,24,B0047,3,4,00004.csv,NaN,0.053192,0.164734,...,NaN,NaN,NaN,NaN,NaN,NaN,1.107108,5.319186e-02,1.647340e-01,2
4,discharge,NaN,4,B0047,4,5,00005.csv,1.524366,NaN,NaN,...,11.314903,3.476154,2.477662,4.001180,-0.995462,-0.992670,1.524366,-3.564623e+13,7.563341e+13,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,charge,NaN,4,B0047,95,96,00096.csv,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.374426,5.623245e+09,-1.193128e+10,39
96,discharge,NaN,4,B0047,96,97,00097.csv,1.199911,NaN,NaN,...,11.363889,3.393144,2.499812,3.994706,-0.994525,-0.991891,1.199911,5.623245e+09,-1.193128e+10,39
97,impedance,NaN,24,B0047,97,98,00098.csv,NaN,0.067098,0.230396,...,NaN,NaN,NaN,NaN,NaN,NaN,1.352227,6.709824e-02,2.303958e-01,40
98,charge,NaN,4,B0047,98,99,00099.csv,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.372326,2.242662e+08,-4.758433e+08,40


In [20]:
import pandas as pd
import numpy as np

# ================== CONFIG ==================
INPUT_METADATA_PATH  = "/kaggle/working/metadata_with_cycle_no.csv"
OUTPUT_METADATA_PATH = "/kaggle/working/metadata_with_soh_all_factors.csv"
# ============================================

# Load metadata (must already have cycle_no)
metadata = pd.read_csv(INPUT_METADATA_PATH)

# ------------ basic checks ------------
if "battery_id" not in metadata.columns or "cycle_no" not in metadata.columns:
    raise ValueError("metadata must contain 'battery_id' and 'cycle_no' columns.")

# Ensure start_time is datetime (for age)
if "start_time" in metadata.columns:
    metadata["start_time"] = pd.to_datetime(metadata["start_time"], errors="coerce")
else:
    metadata["start_time"] = pd.NaT

# ============================================================
# 1) Define capacity_now using all available capacity info
#    Priority:
#      1) Capacity (measured)
#      2) Capacity_pred (model)
#      3) bms_disc_capacity_ah (from BMS)
# ============================================================

capacity_now = pd.Series(np.nan, index=metadata.index)

if "Capacity" in metadata.columns:
    capacity_now = metadata["Capacity"].copy()

if "Capacity_pred" in metadata.columns:
    capacity_now = capacity_now.fillna(metadata["Capacity_pred"])

if "bms_disc_capacity_ah" in metadata.columns:
    capacity_now = capacity_now.fillna(metadata["bms_disc_capacity_ah"])

# Remove non-positive / invalid capacities
capacity_now[capacity_now <= 0] = np.nan
metadata["capacity_now"] = capacity_now

# ============================================================
# 2) Build reference values from FIRST CYCLE (cycle_no == 1)
#    for each battery_id:
#      - capacity_ref
#      - bms_disc_capacity_ref
#      - bms_charge_capacity_ref
#      - Re_ref, Rct_ref
#      - temp_ref, voltage_ref, current_ref
#      - start_time_ref (for age)
# ============================================================

ref_agg = {}

# Always include these if present
ref_agg["capacity_now"] = "mean"

for col in [
    "bms_disc_capacity_ah",
    "bms_charge_capacity_ah",
    "Re_pred",
    "Rct_pred",
    "bms_mean_temp",
    "bms_max_temp",
    "bms_mean_voltage",
    "bms_max_voltage",
    "bms_mean_current",
    "bms_max_current",
]:
    if col in metadata.columns:
        ref_agg[col] = "mean"

# start_time: earliest in cycle 1
if "start_time" in metadata.columns:
    ref_agg["start_time"] = "min"

ref_df = (
    metadata[metadata["cycle_no"] == 1]
    .groupby("battery_id", as_index=True)
    .agg(ref_agg)
    .rename(columns=lambda c: c + "_ref")
)

# Merge reference info back into metadata (left merge preserves row order)
metadata = metadata.merge(ref_df, on="battery_id", how="left", sort=False)

# For convenience: rename some key reference columns
metadata.rename(
    columns={
        "capacity_now_ref": "capacity_ref",
        "bms_disc_capacity_ah_ref": "disc_cap_ref",
        "bms_charge_capacity_ah_ref": "charge_cap_ref",
        "Re_pred_ref": "Re_ref",
        "Rct_pred_ref": "Rct_ref",
        "bms_mean_temp_ref": "temp_mean_ref",
        "bms_max_temp_ref": "temp_max_ref",
        "bms_mean_voltage_ref": "volt_mean_ref",
        "bms_max_voltage_ref": "volt_max_ref",
        "bms_mean_current_ref": "curr_mean_ref",
        "bms_max_current_ref": "curr_max_ref",
        "start_time_ref": "start_time_ref",
    },
    inplace=True,
)

# ============================================================
# 3) Compute individual SoH indicators (relative 0–1, clipped)
#    - capacity-based
#    - discharge / charge capacity-based
#    - Re / Rct-based (impedance)
#    - operating-condition-based (temp / current / voltage)
# ============================================================

# Helper to compute a ratio safely and clip to [0, 1]
def safe_ratio(numer, denom):
    ratio = numer / denom
    ratio = ratio.replace([np.inf, -np.inf], np.nan)
    return ratio.clip(lower=0.0, upper=1.0)

# --- Capacity SoH ---
metadata["SoH_capacity_rel"] = np.nan
valid_cap_mask = metadata["capacity_now"].notna() & metadata["capacity_ref"].notna() & (metadata["capacity_ref"] > 0)
metadata.loc[valid_cap_mask, "SoH_capacity_rel"] = safe_ratio(
    metadata.loc[valid_cap_mask, "capacity_now"],
    metadata.loc[valid_cap_mask, "capacity_ref"],
)

# --- Discharge capacity SoH ---
if "bms_disc_capacity_ah" in metadata.columns and "disc_cap_ref" in metadata.columns:
    metadata["SoH_disc_rel"] = np.nan
    mask = metadata["bms_disc_capacity_ah"].notna() & metadata["disc_cap_ref"].notna() & (metadata["disc_cap_ref"] > 0)
    metadata.loc[mask, "SoH_disc_rel"] = safe_ratio(
        metadata.loc[mask, "bms_disc_capacity_ah"],
        metadata.loc[mask, "disc_cap_ref"],
    )

# --- Charge capacity SoH ---
if "bms_charge_capacity_ah" in metadata.columns and "charge_cap_ref" in metadata.columns:
    metadata["SoH_charge_rel"] = np.nan
    mask = metadata["bms_charge_capacity_ah"].notna() & metadata["charge_cap_ref"].notna() & (metadata["charge_cap_ref"] > 0)
    metadata.loc[mask, "SoH_charge_rel"] = safe_ratio(
        metadata.loc[mask, "bms_charge_capacity_ah"],
        metadata.loc[mask, "charge_cap_ref"],
    )

# --- Resistance-based SoH (Re, Rct) ---
if "Re_pred" in metadata.columns and "Re_ref" in metadata.columns:
    metadata["SoH_Re"] = np.nan
    mask = metadata["Re_pred"].notna() & metadata["Re_ref"].notna() & (metadata["Re_pred"] > 0)
    # SoH_Re = Re_ref / Re_pred  (higher Re => lower SoH)
    metadata.loc[mask, "SoH_Re"] = safe_ratio(
        metadata.loc[mask, "Re_ref"],
        metadata.loc[mask, "Re_pred"],
    )

if "Rct_pred" in metadata.columns and "Rct_ref" in metadata.columns:
    metadata["SoH_Rct"] = np.nan
    mask = metadata["Rct_pred"].notna() & metadata["Rct_ref"].notna() & (metadata["Rct_pred"] > 0)
    metadata.loc[mask, "SoH_Rct"] = safe_ratio(
        metadata.loc[mask, "Rct_ref"],
        metadata.loc[mask, "Rct_pred"],
    )

# --- Temperature-based "health" (cooler is better) ---
if "bms_mean_temp" in metadata.columns and "temp_mean_ref" in metadata.columns:
    metadata["SoH_temp_rel"] = np.nan
    mask = metadata["bms_mean_temp"].notna() & metadata["temp_mean_ref"].notna() & (metadata["bms_mean_temp"] > 0)
    # SoH_temp = temp_ref / temp_now (if you run hotter than reference, SoH_temp < 1)
    metadata.loc[mask, "SoH_temp_rel"] = safe_ratio(
        metadata.loc[mask, "temp_mean_ref"],
        metadata.loc[mask, "bms_mean_temp"],
    )

# --- Current-based "health" (lower current stress is better) ---
if "bms_mean_current" in metadata.columns and "curr_mean_ref" in metadata.columns:
    metadata["SoH_current_rel"] = np.nan
    mask = metadata["bms_mean_current"].notna() & metadata["curr_mean_ref"].notna() & (metadata["bms_mean_current"] != 0)
    # Use absolute current (magnitude of stress)
    numer = metadata.loc[mask, "curr_mean_ref"].abs()
    denom = metadata.loc[mask, "bms_mean_current"].abs()
    metadata.loc[mask, "SoH_current_rel"] = safe_ratio(numer, denom)

# --- Voltage-based "health" (lower max voltage stress is better) ---
if "bms_max_voltage" in metadata.columns and "volt_max_ref" in metadata.columns:
    metadata["SoH_voltage_rel"] = np.nan
    mask = metadata["bms_max_voltage"].notna() & metadata["volt_max_ref"].notna() & (metadata["bms_max_voltage"] > 0)
    # Assume staying near reference or lower is better:
    metadata.loc[mask, "SoH_voltage_rel"] = safe_ratio(
        metadata.loc[mask, "volt_max_ref"],
        metadata.loc[mask, "bms_max_voltage"],
    )

# ------------------------------------------------------------
# Convert the main capacity SoH into percent for convenience
# ------------------------------------------------------------
metadata["SoH_capacity_percent"] = (metadata["SoH_capacity_rel"] * 100.0).clip(lower=0.0, upper=100.0)

# ============================================================
# 4) Age features (time since first reference cycle)
# ============================================================
if "start_time_ref" not in metadata.columns:
    # If not present (older pandas agg), create from start_time_ref column name we used
    if "start_time_ref_ref" in metadata.columns:
        metadata["start_time_ref"] = metadata["start_time_ref_ref"]
    else:
        metadata["start_time_ref"] = pd.NaT

age_timedelta = metadata["start_time"] - metadata["start_time_ref"]
metadata["age_s"] = age_timedelta.dt.total_seconds()
metadata["age_days"] = metadata["age_s"] / (3600.0 * 24.0)

# ============================================================
# 5) Combine all available SoH components into one scalar
#    SoH_combined_rel in [0, 1], SoH_combined_percent in [0, 100]
# ============================================================

# Define base weights for each component (can be tuned)
component_weights = {
    "SoH_capacity_rel": 0.50,  # main driver
    "SoH_disc_rel":     0.10,
    "SoH_charge_rel":   0.05,
    "SoH_Re":           0.10,
    "SoH_Rct":          0.10,
    "SoH_temp_rel":     0.05,
    "SoH_current_rel":  0.05,
    "SoH_voltage_rel":  0.05,
}

soh_cols_present = [c for c in component_weights.keys() if c in metadata.columns]

def combine_soh_row(row):
    num = 0.0
    denom = 0.0
    for col in soh_cols_present:
        val = row[col]
        if pd.notna(val):
            w = component_weights[col]
            num += w * val
            denom += w
    if denom == 0.0:
        return np.nan
    return num / denom

metadata["SoH_combined_rel"] = metadata.apply(combine_soh_row, axis=1)
metadata["SoH_combined_rel"] = metadata["SoH_combined_rel"].clip(lower=0.0, upper=1.0)
metadata["SoH_combined_percent"] = (metadata["SoH_combined_rel"] * 100.0).clip(lower=0.0, upper=100.0)

# ============================================================
# 6) Save updated metadata
# ============================================================
metadata.to_csv(OUTPUT_METADATA_PATH, index=False)
print("Saved metadata with multi-factor SoH to:", OUTPUT_METADATA_PATH)


/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)


Saved metadata with multi-factor SoH to: /kaggle/working/metadata_with_soh_all_factors.csv


In [21]:
df453 = pd.read_csv("/kaggle/working/metadata_with_soh_all_factors.csv")
df453

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct,...,SoH_Re,SoH_Rct,SoH_temp_rel,SoH_current_rel,SoH_voltage_rel,SoH_capacity_percent,age_s,age_days,SoH_combined_rel,SoH_combined_percent
0,discharge,NaN,4,B0047,0,1,00001.csv,1.674305,NaN,NaN,...,NaN,1.000000,1.000000,1.000000,1.000000,100.000000,NaN,NaN,1.000000,100.000000
1,impedance,NaN,24,B0047,1,2,00002.csv,NaN,0.056058,0.200970,...,0.0,1.000000,NaN,NaN,NaN,66.051302,NaN,NaN,0.614652,61.465216
2,charge,NaN,4,B0047,2,3,00003.csv,NaN,NaN,NaN,...,0.0,NaN,NaN,NaN,NaN,83.740870,NaN,NaN,0.697841,69.784058
3,impedance,NaN,24,B0047,3,4,00004.csv,NaN,0.053192,0.164734,...,0.0,1.000000,NaN,NaN,NaN,66.123422,NaN,NaN,0.615167,61.516730
4,discharge,NaN,4,B0047,4,5,00005.csv,1.524366,NaN,NaN,...,NaN,0.079343,0.996468,0.999887,1.000000,91.044729,NaN,NaN,0.827947,82.794670
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7560,impedance,NaN,24,B0055,247,7561,07561.csv,NaN,0.096809,0.154897,...,1.0,0.000000,NaN,NaN,NaN,100.000000,NaN,NaN,0.857143,85.714286
7561,discharge,NaN,4,B0055,248,7562,07562.csv,1.020138,NaN,NaN,...,NaN,0.000000,1.000000,1.000000,0.858775,100.000000,NaN,NaN,0.874046,87.404559
7562,charge,NaN,4,B0055,249,7563,07563.csv,NaN,NaN,NaN,...,NaN,0.000000,NaN,NaN,NaN,100.000000,NaN,NaN,0.833333,83.333333
7563,discharge,NaN,4,B0055,250,7564,07564.csv,0.990759,NaN,NaN,...,NaN,0.000000,1.000000,1.000000,0.859595,100.000000,NaN,NaN,0.874094,87.409385


In [22]:
import pandas as pd
import numpy as np

# ================== CONFIG ==================
INPUT_METADATA_PATH  = "/kaggle/working/metadata_with_cycle_no.csv"
OUTPUT_METADATA_PATH = "/kaggle/working/metadata_with_soh_no_impedance.csv"
# ============================================

# Load metadata (must already have cycle_no)
metadata = pd.read_csv(INPUT_METADATA_PATH)

# ------------ basic checks ------------
if "battery_id" not in metadata.columns or "cycle_no" not in metadata.columns:
    raise ValueError("metadata must contain 'battery_id' and 'cycle_no' columns.")

# Ensure start_time is datetime (for age)
if "start_time" in metadata.columns:
    metadata["start_time"] = pd.to_datetime(metadata["start_time"], errors="coerce")
else:
    metadata["start_time"] = pd.NaT

# ============================================================
# 1) Define capacity_now using all available capacity info
#    Priority:
#      1) Capacity (measured)
#      2) Capacity_pred (model)
#      3) bms_disc_capacity_ah (from BMS)
# ============================================================

capacity_now = pd.Series(np.nan, index=metadata.index)

if "Capacity" in metadata.columns:
    capacity_now = metadata["Capacity"].copy()

if "Capacity_pred" in metadata.columns:
    capacity_now = capacity_now.fillna(metadata["Capacity_pred"])

if "bms_disc_capacity_ah" in metadata.columns:
    capacity_now = capacity_now.fillna(metadata["bms_disc_capacity_ah"])

# Remove non-positive / invalid capacities
capacity_now[capacity_now <= 0] = np.nan
metadata["capacity_now"] = capacity_now

# ============================================================
# 2) Build reference values from FIRST CYCLE (cycle_no == 1)
#    for each battery_id:
#      - capacity_ref
#      - disc_cap_ref, charge_cap_ref
#      - temp_ref, voltage_ref, current_ref
#      - start_time_ref (for age)
# ============================================================

ref_agg = {}

# Always include this
ref_agg["capacity_now"] = "mean"

for col in [
    "bms_disc_capacity_ah",
    "bms_charge_capacity_ah",
    "bms_mean_temp",
    "bms_max_temp",
    "bms_mean_voltage",
    "bms_max_voltage",
    "bms_mean_current",
    "bms_max_current",
]:
    if col in metadata.columns:
        ref_agg[col] = "mean"

if "start_time" in metadata.columns:
    ref_agg["start_time"] = "min"

ref_df = (
    metadata[metadata["cycle_no"] == 1]
    .groupby("battery_id", as_index=True)
    .agg(ref_agg)
    .rename(columns=lambda c: c + "_ref")
)

metadata = metadata.merge(ref_df, on="battery_id", how="left", sort=False)

metadata.rename(
    columns={
        "capacity_now_ref": "capacity_ref",
        "bms_disc_capacity_ah_ref": "disc_cap_ref",
        "bms_charge_capacity_ah_ref": "charge_cap_ref",
        "bms_mean_temp_ref": "temp_mean_ref",
        "bms_max_temp_ref": "temp_max_ref",
        "bms_mean_voltage_ref": "volt_mean_ref",
        "bms_max_voltage_ref": "volt_max_ref",
        "bms_mean_current_ref": "curr_mean_ref",
        "bms_max_current_ref": "curr_max_ref",
        "start_time_ref": "start_time_ref",
    },
    inplace=True,
)

# ============================================================
# 3) Compute individual SoH indicators (relative 0–1, clipped)
#    - capacity-based
#    - discharge / charge capacity-based
#    - temp / current / voltage-based “stress” indicators
# ============================================================

def safe_ratio(numer, denom):
    ratio = numer / denom
    ratio = ratio.replace([np.inf, -np.inf], np.nan)
    return ratio.clip(lower=0.0, upper=1.0)

# --- Capacity SoH ---
metadata["SoH_capacity_rel"] = np.nan
valid_cap_mask = (
    metadata["capacity_now"].notna()
    & metadata["capacity_ref"].notna()
    & (metadata["capacity_ref"] > 0)
)
metadata.loc[valid_cap_mask, "SoH_capacity_rel"] = safe_ratio(
    metadata.loc[valid_cap_mask, "capacity_now"],
    metadata.loc[valid_cap_mask, "capacity_ref"],
)

# --- Discharge capacity SoH ---
if "bms_disc_capacity_ah" in metadata.columns and "disc_cap_ref" in metadata.columns:
    metadata["SoH_disc_rel"] = np.nan
    mask = (
        metadata["bms_disc_capacity_ah"].notna()
        & metadata["disc_cap_ref"].notna()
        & (metadata["disc_cap_ref"] > 0)
    )
    metadata.loc[mask, "SoH_disc_rel"] = safe_ratio(
        metadata.loc[mask, "bms_disc_capacity_ah"],
        metadata.loc[mask, "disc_cap_ref"],
    )

# --- Charge capacity SoH ---
if "bms_charge_capacity_ah" in metadata.columns and "charge_cap_ref" in metadata.columns:
    metadata["SoH_charge_rel"] = np.nan
    mask = (
        metadata["bms_charge_capacity_ah"].notna()
        & metadata["charge_cap_ref"].notna()
        & (metadata["charge_cap_ref"] > 0)
    )
    metadata.loc[mask, "SoH_charge_rel"] = safe_ratio(
        metadata.loc[mask, "bms_charge_capacity_ah"],
        metadata.loc[mask, "charge_cap_ref"],
    )

# --- Temperature-based “health” (cooler than ref is better) ---
if "bms_mean_temp" in metadata.columns and "temp_mean_ref" in metadata.columns:
    metadata["SoH_temp_rel"] = np.nan
    mask = (
        metadata["bms_mean_temp"].notna()
        & metadata["temp_mean_ref"].notna()
        & (metadata["bms_mean_temp"] > 0)
    )
    metadata.loc[mask, "SoH_temp_rel"] = safe_ratio(
        metadata.loc[mask, "temp_mean_ref"],
        metadata.loc[mask, "bms_mean_temp"],
    )

# --- Current-based “health” (lower magnitude than ref is better) ---
if "bms_mean_current" in metadata.columns and "curr_mean_ref" in metadata.columns:
    metadata["SoH_current_rel"] = np.nan
    mask = (
        metadata["bms_mean_current"].notna()
        & metadata["curr_mean_ref"].notna()
        & (metadata["bms_mean_current"] != 0)
    )
    numer = metadata.loc[mask, "curr_mean_ref"].abs()
    denom = metadata.loc[mask, "bms_mean_current"].abs()
    metadata.loc[mask, "SoH_current_rel"] = safe_ratio(numer, denom)

# --- Voltage-based “health” (max voltage above ref is more stress) ---
if "bms_max_voltage" in metadata.columns and "volt_max_ref" in metadata.columns:
    metadata["SoH_voltage_rel"] = np.nan
    mask = (
        metadata["bms_max_voltage"].notna()
        & metadata["volt_max_ref"].notna()
        & (metadata["bms_max_voltage"] > 0)
    )
    metadata.loc[mask, "SoH_voltage_rel"] = safe_ratio(
        metadata.loc[mask, "volt_max_ref"],
        metadata.loc[mask, "bms_max_voltage"],
    )

# Capacity SoH in percent
metadata["SoH_capacity_percent"] = (
    metadata["SoH_capacity_rel"] * 100.0
).clip(lower=0.0, upper=100.0)

# ============================================================
# 4) Age features (time since first reference cycle)
# ============================================================
age_timedelta = metadata["start_time"] - metadata["start_time_ref"]
metadata["age_s"] = age_timedelta.dt.total_seconds()
metadata["age_days"] = metadata["age_s"] / (3600.0 * 24.0)

# ============================================================
# 5) Combine SoH components into one scalar (no Re/Rct)
#    SoH_combined_rel in [0, 1], SoH_combined_percent in [0, 100]
# ============================================================

component_weights = {
    "SoH_capacity_rel": 0.60,  # main
    "SoH_disc_rel":     0.15,
    "SoH_charge_rel":   0.05,
    "SoH_temp_rel":     0.05,
    "SoH_current_rel":  0.10,
    "SoH_voltage_rel":  0.05,
}

soh_cols_present = [c for c in component_weights.keys() if c in metadata.columns]

def combine_soh_row(row):
    num = 0.0
    denom = 0.0
    for col in soh_cols_present:
        val = row[col]
        if pd.notna(val):
            w = component_weights[col]
            num += w * val
            denom += w
    if denom == 0.0:
        return np.nan
    return num / denom

metadata["SoH_combined_rel_no_imp"] = metadata.apply(combine_soh_row, axis=1)
metadata["SoH_combined_rel_no_imp"] = metadata["SoH_combined_rel_no_imp"].clip(lower=0.0, upper=1.0)
metadata["SoH_combined_percent_no_imp"] = (
    metadata["SoH_combined_rel_no_imp"] * 100.0
).clip(lower=0.0, upper=100.0)

# ============================================================
# 6) Save updated metadata
# ============================================================
metadata.to_csv(OUTPUT_METADATA_PATH, index=False)
print("Saved metadata with SoH (no impedance) to:", OUTPUT_METADATA_PATH)


/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)


Saved metadata with SoH (no impedance) to: /kaggle/working/metadata_with_soh_no_impedance.csv


In [23]:
df65 = pd.read_csv("/kaggle/working/data_with_soc/00001.csv")
df65

,Voltage_measured,Current_measured,Temperature_measured,Current_load,Voltage_load,Time,delta_t,I_clean,phase,delta_Ah,cum_Ah,SoC_unclipped,SoC,SoC_percent
0,4.246711,0.000252,6.212696,0.0002,0.000,0.000,0.000,0.000000,impedance,0.000000,0.000000,0.000000,0.0,0.0
1,4.246764,-0.001411,6.234019,0.0002,4.262,9.360,9.360,-0.001411,discharge,-0.000004,-0.000004,-0.000002,0.0,0.0
2,4.039277,-0.995093,6.250255,1.0000,3.465,23.281,13.921,-0.995093,discharge,-0.003848,-0.003852,-0.002300,0.0,0.0
3,4.019506,-0.996731,6.302176,1.0000,3.451,36.406,13.125,-0.996731,discharge,-0.003634,-0.007486,-0.004471,0.0,0.0
4,4.004763,-0.992845,6.361645,1.0000,3.438,49.625,13.219,-0.992845,discharge,-0.003646,-0.011131,-0.006648,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
485,3.303251,-0.001760,9.662331,0.0004,0.000,6382.063,13.641,-0.001760,discharge,-0.000007,-1.705952,-1.018902,0.0,0.0
486,3.310303,-0.000756,9.390489,0.0002,0.000,6395.547,13.484,0.000000,impedance,0.000000,-1.705952,-1.018902,0.0,0.0
487,3.317351,-0.003318,9.137008,0.0002,0.000,6409.063,13.516,-0.003318,discharge,-0.000012,-1.705964,-1.018909,0.0,0.0
488,3.323387,-0.002291,8.972806,0.0002,0.000,6422.625,13.562,-0.002291,discharge,-0.000009,-1.705973,-1.018914,0.0,0.0


In [24]:
import pandas as pd
import numpy as np

# ================== CONFIG ==================
INPUT_METADATA_PATH  = "/kaggle/working/metadata_with_soh.csv"  # or your latest SoH file
OUTPUT_METADATA_PATH = "/kaggle/working/metadata_with_rul.csv"

SOH_COLUMN_PRIORITY = ["SoH_combined_rel", "SoH_capacity_rel"]  # first available is used
SOH_EOL_THRESHOLD   = 0.80  # EOL at 80% SoH
# ============================================

# Load metadata with SoH
metadata = pd.read_csv(INPUT_METADATA_PATH)

# --------- Basic checks ---------
for col in ["battery_id", "cycle_no"]:
    if col not in metadata.columns:
        raise ValueError(f"Required column '{col}' not found in metadata.")

# Time columns
if "start_time" in metadata.columns:
    metadata["start_time"] = pd.to_datetime(metadata["start_time"], errors="coerce")
else:
    metadata["start_time"] = pd.NaT

# If age_days exists from previous steps, keep it; else compute it now
if "age_days" not in metadata.columns:
    # Use min(start_time) per battery as reference
    ref_start = (
        metadata.groupby("battery_id")["start_time"]
        .transform("min")
    )
    age_timedelta = metadata["start_time"] - ref_start
    metadata["age_s"] = age_timedelta.dt.total_seconds()
    metadata["age_days"] = metadata["age_s"] / (3600.0 * 24.0)
else:
    # ensure numeric
    metadata["age_days"] = pd.to_numeric(metadata["age_days"], errors="coerce")

# Usage time per row from BMS if available
if "bms_active_time_s" in metadata.columns:
    metadata["usage_time_s"] = pd.to_numeric(metadata["bms_active_time_s"], errors="coerce")
else:
    # Fallback: approximate usage_time_s as difference in age_s between cycles
    if "age_s" not in metadata.columns:
        # compute age_s if missing
        ref_start = (
            metadata.groupby("battery_id")["start_time"]
            .transform("min")
        )
        age_timedelta = metadata["start_time"] - ref_start
        metadata["age_s"] = age_timedelta.dt.total_seconds()
    metadata["usage_time_s"] = (
        metadata
        .sort_values(["battery_id", "cycle_no"])
        .groupby("battery_id")["age_s"]
        .diff()
        .fillna(0.0)
    )

# Cumulative usage time per battery (sum of usage_time_s)
metadata = metadata.sort_values(["battery_id", "cycle_no"], ignore_index=True)
metadata["cum_usage_time_s"] = (
    metadata
    .groupby("battery_id")["usage_time_s"]
    .cumsum()
)

# --------- Choose SoH metric to base RUL on ---------
soh_col = None
for c in SOH_COLUMN_PRIORITY:
    if c in metadata.columns:
        soh_col = c
        break

if soh_col is None:
    raise ValueError(f"None of the specified SoH columns {SOH_COLUMN_PRIORITY} found in metadata.")

print(f"Using SoH column for RUL: {soh_col}")

# Ensure SoH is numeric and clipped to [0,1]
metadata[soh_col] = pd.to_numeric(metadata[soh_col], errors="coerce")
metadata[soh_col] = metadata[soh_col].clip(lower=0.0, upper=1.0)

# ====================================================
# Helper: compute RUL per battery_id with simple linear
# degradation model, both vs cycle_no and vs time.
# ====================================================

def compute_rul_for_battery(df_bat: pd.DataFrame) -> pd.DataFrame:
    """
    df_bat: all rows for one battery_id, sorted by cycle_no.

    Adds per-row:
      - degr_rate_per_cycle
      - EOL_cycle_est
      - RUL_cycles

      - degr_rate_per_day
      - EOL_age_days_est
      - RUL_age_days

      - degr_rate_per_usage_s
      - EOL_usage_time_s_est
      - RUL_usage_time_s, RUL_usage_time_h
    """
    df_bat = df_bat.sort_values("cycle_no").copy()

    # --- basic arrays ---
    cycles = df_bat["cycle_no"].to_numpy(dtype=float)
    soh    = df_bat[soh_col].to_numpy(dtype=float)
    age_days = df_bat["age_days"].to_numpy(dtype=float)
    cum_usage = df_bat["cum_usage_time_s"].to_numpy(dtype=float)

    # Filter out NaN soh and degenerate cases
    valid_mask = ~np.isnan(soh) & ~np.isnan(cycles)
    if valid_mask.sum() < 2:
        # Not enough points to estimate trend → no RUL
        df_bat["degr_rate_per_cycle"]    = np.nan
        df_bat["EOL_cycle_est"]          = np.nan
        df_bat["RUL_cycles"]             = np.nan
        df_bat["degr_rate_per_day"]      = np.nan
        df_bat["EOL_age_days_est"]       = np.nan
        df_bat["RUL_age_days"]           = np.nan
        df_bat["degr_rate_per_usage_s"]  = np.nan
        df_bat["EOL_usage_time_s_est"]   = np.nan
        df_bat["RUL_usage_time_s"]       = np.nan
        df_bat["RUL_usage_time_h"]       = np.nan
        return df_bat

    # We’ll just use first and last valid points for a simple slope (robust & cheap)
    first_idx = np.where(valid_mask)[0][0]
    last_idx  = np.where(valid_mask)[0][-1]

    cyc1, cycN = cycles[first_idx], cycles[last_idx]
    soh1, sohN = soh[first_idx], soh[last_idx]

    # ---- Degradation rate per cycle ----
    if cycN > cyc1:
        degr_cycle = (sohN - soh1) / (cycN - cyc1)  # usually negative
    else:
        degr_cycle = np.nan

    # Avoid division by zero or positive slope (no degradation or growing)
    if np.isnan(degr_cycle) or degr_cycle >= 0:
        EOL_cycle_est = np.nan
    else:
        # Linear model: soh ≈ soh1 + degr_cycle * (cycle - cyc1)
        # EOL when soh = SOH_EOL_THRESHOLD:
        # SOH_EOL = soh1 + degr_cycle * (EOL_cycle - cyc1)
        # => EOL_cycle = cyc1 + (SOH_EOL - soh1) / degr_cycle
        EOL_cycle_est = cyc1 + (SOH_EOL_THRESHOLD - soh1) / degr_cycle

    # RUL in cycles for each row
    df_bat["degr_rate_per_cycle"] = degr_cycle
    df_bat["EOL_cycle_est"] = EOL_cycle_est
    df_bat["RUL_cycles"] = EOL_cycle_est - df_bat["cycle_no"]
    df_bat.loc[df_bat["RUL_cycles"] < 0, "RUL_cycles"] = 0.0

    # ---- Degradation rate per age_day ----
    valid_time_mask = valid_mask & ~np.isnan(age_days)
    if valid_time_mask.sum() >= 2:
        t1 = age_days[valid_time_mask][0]
        tN = age_days[valid_time_mask][-1]
        soh1_t = soh[valid_time_mask][0]
        sohN_t = soh[valid_time_mask][-1]

        if tN > t1:
            degr_day = (sohN_t - soh1_t) / (tN - t1)
        else:
            degr_day = np.nan

        if np.isnan(degr_day) or degr_day >= 0:
            EOL_age_days_est = np.nan
        else:
            EOL_age_days_est = t1 + (SOH_EOL_THRESHOLD - soh1_t) / degr_day
    else:
        degr_day = np.nan
        EOL_age_days_est = np.nan

    df_bat["degr_rate_per_day"] = degr_day
    df_bat["EOL_age_days_est"] = EOL_age_days_est
    df_bat["RUL_age_days"] = EOL_age_days_est - df_bat["age_days"]
    df_bat.loc[df_bat["RUL_age_days"] < 0, "RUL_age_days"] = 0.0

    # ---- Degradation rate per cumulative usage time (seconds) ----
    valid_usage_mask = valid_mask & ~np.isnan(cum_usage)
    if valid_usage_mask.sum() >= 2:
        u1 = cum_usage[valid_usage_mask][0]
        uN = cum_usage[valid_usage_mask][-1]
        soh1_u = soh[valid_usage_mask][0]
        sohN_u = soh[valid_usage_mask][-1]

        if uN > u1:
            degr_usage = (sohN_u - soh1_u) / (uN - u1)
        else:
            degr_usage = np.nan

        if np.isnan(degr_usage) or degr_usage >= 0:
            EOL_usage_est = np.nan
        else:
            EOL_usage_est = u1 + (SOH_EOL_THRESHOLD - soh1_u) / degr_usage
    else:
        degr_usage = np.nan
        EOL_usage_est = np.nan

    df_bat["degr_rate_per_usage_s"] = degr_usage
    df_bat["EOL_usage_time_s_est"] = EOL_usage_est
    df_bat["RUL_usage_time_s"] = EOL_usage_est - df_bat["cum_usage_time_s"]
    df_bat.loc[df_bat["RUL_usage_time_s"] < 0, "RUL_usage_time_s"] = 0.0
    df_bat["RUL_usage_time_h"] = df_bat["RUL_usage_time_s"] / 3600.0

    return df_bat


# ====================================================
# Apply RUL computation per battery_id
# ====================================================
metadata_with_rul = (
    metadata
    .groupby("battery_id", group_keys=False)
    .apply(compute_rul_for_battery)
)

# Keep original row order if you care about it
metadata_with_rul = metadata_with_rul.sort_values(["battery_id", "cycle_no"]).reset_index(drop=True)

# Save
metadata_with_rul.to_csv(OUTPUT_METADATA_PATH, index=False)
print("Saved metadata with RUL to:", OUTPUT_METADATA_PATH)


Using SoH column for RUL: SoH_combined_rel


/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less_equal
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in less
  return op(a, b)
/usr/local/lib/python3.11/dist-packages/pandas/core/computation/expressions.py:

Saved metadata with RUL to: /kaggle/working/metadata_with_rul.csv


In [25]:
df65 = pd.read_csv("/kaggle/working/metadata_with_rul.csv")
df65

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pan

,type,start_time,ambient_temperature,battery_id,test_id,uid,filename,Capacity,Re,Rct,...,degr_rate_per_cycle,EOL_cycle_est,RUL_cycles,degr_rate_per_day,EOL_age_days_est,RUL_age_days,degr_rate_per_usage_s,EOL_usage_time_s_est,RUL_usage_time_s,RUL_usage_time_h
0,charge,NaN,24,B0005,0,5121,05121.csv,NaN,NaN,NaN,...,-0.001099,22.246975,21.246975,NaN,NaN,NaN,-0.000001,198923.600497,NaN,NaN
1,discharge,NaN,24,B0005,1,5122,05122.csv,1.856487,NaN,NaN,...,-0.001099,22.246975,21.246975,NaN,NaN,NaN,-0.000001,198923.600497,198923.600497,55.256556
2,charge,NaN,24,B0005,2,5123,05123.csv,NaN,NaN,NaN,...,-0.001099,22.246975,20.246975,NaN,NaN,NaN,-0.000001,198923.600497,NaN,NaN
3,discharge,NaN,24,B0005,3,5124,05124.csv,1.846327,NaN,NaN,...,-0.001099,22.246975,20.246975,NaN,NaN,NaN,-0.000001,198923.600497,198923.600497,55.256556
4,charge,NaN,24,B0005,4,5125,05125.csv,NaN,NaN,NaN,...,-0.001099,22.246975,19.246975,NaN,NaN,NaN,-0.000001,198923.600497,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7560,impedance,NaN,24,B0056,247,7309,07309.csv,NaN,0.102677,0.170394,...,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN
7561,discharge,NaN,4,B0056,248,7310,07310.csv,1.137273,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN
7562,discharge,NaN,4,B0056,250,7312,07312.csv,1.129059,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN
7563,charge,NaN,4,B0056,249,7311,07311.csv,NaN,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000,NaN,NaN,NaN
